# Visualizing Carbon Dioxide Levels — Solution

Complete, annotated solution with **alternate implementations**, **more practice**, **parameterised simulation**, and audience notes.

**Flowchart**

![Flowchart](co2_levels_flowchart.png)


In [ ]:
# Setup
library(readr)
library(dplyr)
library(ggplot2)
library(tidyr)
library(gridExtra)
options(scipen = 10000)


## Part 1 — NOAA ice-core record

In [ ]:
# Tasks 1–2
noaa_data <- read_csv("data/carbon_dioxide_levels.csv", show_col_types = FALSE)
head(noaa_data)
# Age_yrBP = years before 1950 (BP), CO2_ppmv = parts per million by volume
summary(noaa_data)
cat("Rows:", nrow(noaa_data), "  Max CO2:", max(noaa_data$CO2_ppmv, na.rm = TRUE), "ppmv\n")


In [ ]:
# Tasks 3–8 — full NOAA line chart with reversed x-axis
noaa_viz <- ggplot(data = noaa_data, aes(x = Age_yrBP, y = CO2_ppmv)) +
  geom_line(color = "#2E86AB", linewidth = 0.55) +
  scale_x_reverse() +
  labs(
    title    = "Carbon Dioxide Levels From 8,000 to 136 Years BP",
    subtitle = "From World Data Center for Paleoclimatology and NOAA Paleoclimatology Program",
    x        = "Years Before Today (0=1950)",
    y        = "Carbon Dioxide Level (Parts Per Million)"
  ) +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold", size = 13))

noaa_viz
# ggsave("co2_noaa_line.png", noaa_viz, width = 9, height = 5, dpi = 120)


## Part 2 — IAC last two millennia + historical peak

In [ ]:
# Tasks 9–10
iac_data <- read_csv("data/yearly_co2.csv", show_col_types = FALSE)
head(iac_data)
summary(iac_data[, 1:2])


In [ ]:
# Task 16 — historical maximum from the ice-core series
millennia_max <- max(noaa_data$CO2_ppmv, na.rm = TRUE)
print(millennia_max)   # 298.6 ppmv (occurs in an older interglacial)


In [ ]:
# Tasks 11–15 + 17–18 — modern series with reference line
iac_viz <- ggplot(data = iac_data, aes(x = year, y = data_mean_global)) +
  geom_line(color = "#E67E22", linewidth = 0.7) +
  geom_hline(
    aes(yintercept = millennia_max,
        linetype   = "Historical CO2 Peak (ice-core max)"),
    color     = "#C0392B",
    linewidth = 0.9
  ) +
  scale_linetype_manual(
    values = c("Historical CO2 Peak (ice-core max)" = "dashed"),
    name   = NULL
  ) +
  labs(
    title    = "Carbon Dioxide Levels over Time",
    subtitle = "From Institute for Atmospheric and Climate Science (IAC)",
    x        = "Year",
    y        = "Carbon Dioxide Level (Parts Per Million)"
  ) +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"),
        legend.position = "bottom")

iac_viz
# ggsave("co2_iac_line.png", iac_viz, width = 9, height = 5, dpi = 120)


**Key observation:** Modern values (reaching ~397 ppm by 2014) already surpass the highest concentration recorded in the entire ice-core archive (~298.6 ppm). The acceleration is almost entirely post-1800 / post-1950.


---
## Alternate Code

### A. Base-R equivalent


In [ ]:
# Base R version of the two main plots
par(mfrow = c(2, 1), mar = c(4, 4, 3, 1))
# NOAA (reverse x by plotting -Age)
plot(-noaa_data$Age_yrBP, noaa_data$CO2_ppmv, type = "l", col = "#2E86AB",
     xlab = "Years Before Today (0 = 1950)", ylab = "CO2 (ppmv)",
     main = "NOAA ice-core (base R)")
# IAC + hline
plot(iac_data$year, iac_data$data_mean_global, type = "l", col = "#E67E22",
     xlab = "Year", ylab = "CO2 (ppmv)",
     main = "IAC yearly + historical peak (base R)")
abline(h = millennia_max, col = "#C0392B", lty = 2, lwd = 2)
par(mfrow = c(1, 1))


### B. geom_path + explicit sorting + Holocene-only reference


In [ ]:
# Holocene-only max (last ~11 kyr) is a tighter, more policy-relevant benchmark
holocene <- noaa_data %>% filter(Age_yrBP <= 11000)
holocene_max <- max(holocene$CO2_ppmv, na.rm = TRUE)
cat("Holocene max:", holocene_max, "ppmv\n")

iac_hol <- ggplot(iac_data, aes(x = year, y = data_mean_global)) +
  geom_path(color = "#8E44AD", linewidth = 0.8) +   # path respects row order
  geom_hline(yintercept = holocene_max, linetype = "dashed", color = "red", linewidth = 0.9) +
  annotate("text", x = 200, y = holocene_max + 8,
           label = paste0("Holocene max ≈ ", round(holocene_max, 1), " ppm"),
           color = "red", hjust = 0, size = 3.5) +
  labs(title = "Modern CO2 vs Holocene Maximum",
       subtitle = "Tighter natural baseline than full 800-kyr max",
       x = "Year", y = "CO2 (ppmv)") +
  theme_minimal()
iac_hol


### C. Northern vs Southern Hemisphere (long format)


In [ ]:
iac_long <- iac_data %>%
  select(year, Global = data_mean_global, NH = data_mean_nh, SH = data_mean_sh) %>%
  pivot_longer(-year, names_to = "Series", values_to = "CO2")

ggplot(iac_long %>% filter(year >= 1900),
       aes(x = year, y = CO2, color = Series)) +
  geom_line(linewidth = 0.8) +
  scale_color_manual(values = c(Global = "#2C3E50", NH = "#E74C3C", SH = "#3498DB")) +
  labs(title = "CO2 by Hemisphere (1900–2014)",
       x = "Year", y = "CO2 (ppmv)") +
  theme_minimal() +
  theme(legend.position = "bottom")


---
## More Practice Solutions


In [ ]:
# 1. Zoom 1800–2014 and find first year exceeding ice-core max
modern <- iac_data %>% filter(year >= 1800)
first_exceed <- modern %>% filter(data_mean_global > millennia_max) %>% slice(1)
print(first_exceed)

ggplot(modern, aes(x = year, y = data_mean_global)) +
  geom_line(color = "#8E44AD", linewidth = 1) +
  geom_hline(yintercept = millennia_max, linetype = "dashed", color = "red") +
  geom_vline(xintercept = first_exceed$year, linetype = "dotted", color = "grey40") +
  annotate("text", x = first_exceed$year + 5, y = 320,
           label = paste("First exceedance ~", first_exceed$year), hjust = 0, size = 3.5) +
  labs(title = "Industrial-era rise vs ice-core maximum",
       x = "Year", y = "CO2 (ppmv)") +
  theme_minimal()


In [ ]:
# 2. Rough rates of change
# Modern: 1950–2014
mod_1950 <- iac_data %>% filter(year %in% c(1950, 2014))
rate_mod <- (mod_1950$data_mean_global[2] - mod_1950$data_mean_global[1]) / ((2014-1950)/10)
cat(sprintf("Modern rate ~ %.2f ppm / decade (1950–2014)\n", rate_mod))

# Any deep-time 1-kyr window (example around 100 kyr BP)
window <- noaa_data %>% filter(Age_yrBP > 99000, Age_yrBP < 101000)
if (nrow(window) > 1) {
  rate_paleo <- abs(diff(range(window$CO2_ppmv))) / 1   # per kyr → /10 for decade
  cat(sprintf("Example paleo window range: %.1f ppm over ~1 kyr\n", diff(range(window$CO2_ppmv))))
}


In [ ]:
# 3. Side-by-side panels (common y optional)
p_deep <- noaa_viz + labs(title = "Deep time (reversed BP)") + theme(plot.title = element_text(size = 11))
p_mod  <- iac_viz  + labs(title = "Last two millennia")   + theme(plot.title = element_text(size = 11))
# grid.arrange(p_deep, p_mod, ncol = 1)   # uncomment when running interactively


---
## Simulation Section (parameterised)


In [ ]:
# ---- PARAMETERS ----
NOAA_AGE_CUTOFF   <- 11000    # try 800000 (full) vs 11000 (Holocene)
MODERN_START_YEAR <- 1750
HLINE_SOURCE      <- "holocene"  # "full" | "holocene"
NOISE_SD          <- 0.0         # try 1.5 or 3 for visual noise test
# --------------------

noaa_sub <- noaa_data %>% filter(Age_yrBP <= NOAA_AGE_CUTOFF)
hline_val <- if (HLINE_SOURCE == "holocene") {
  max((noaa_data %>% filter(Age_yrBP <= 11000))$CO2_ppmv, na.rm = TRUE)
} else {
  max(noaa_data$CO2_ppmv, na.rm = TRUE)
}
cat("Using hline value:", hline_val, "ppmv\n")

iac_sim <- iac_data %>%
  filter(year >= MODERN_START_YEAR) %>%
  mutate(CO2_noisy = data_mean_global + rnorm(n(), 0, NOISE_SD))

p_sim <- ggplot(iac_sim, aes(x = year, y = CO2_noisy)) +
  geom_line(color = "#27AE60", linewidth = 0.8) +
  geom_hline(yintercept = hline_val, linetype = "dashed", color = "#C0392B", linewidth = 1) +
  labs(title = paste0("Simulation: start=", MODERN_START_YEAR,
                      " | cutoff=", NOAA_AGE_CUTOFF,
                      " | noise SD=", NOISE_SD),
       subtitle = paste("Reference line =", round(hline_val, 1), "ppmv"),
       x = "Year", y = "CO2 (ppmv)") +
  theme_minimal()
print(p_sim)

# Optional: simple linear projection 2014 → 2050 under recent rate
recent <- iac_data %>% filter(year >= 2000)
fit <- lm(data_mean_global ~ year, data = recent)
future_years <- data.frame(year = 2015:2050)
future_years$pred <- predict(fit, future_years)
cat("Linear extrapolation to 2050 under 2000–2014 trend:\n")
print(tail(future_years, 3))


---
## Audience-adapted key messages

| Audience        | One-sentence takeaway |
|-----------------|-----------------------|
| **Expert**      | Anthropogenic CO₂ has driven atmospheric concentrations beyond the highest levels of the last 800 kyr; the rate of change is 1–2 orders of magnitude faster than typical orbital-scale transitions. |
| **Executive**   | Current CO₂ already exceeds every natural peak in the ice-core record; the rise since 1950 is the dominant feature of the last two millennia. |
| **Nonspecialist** | The red dashed line shows the highest natural CO₂ level we ever measured before industry. Everything above that line is human-caused and new. |

---
## Re-usable template tip
Copy this notebook, replace the two CSV paths and the column-name mappings inside `aes()`, and you have a ready-made deep-time vs modern comparison for any analogous paleo + instrumental series (temperature, CH₄, sea level, etc.).
